In [46]:
# Установка зависимостей (если нужно)
# !pip install tritonclient[grpc] numpy

import numpy as np
import tritonclient.grpc as grpcclient
from tritonclient.grpc import InferInput, InferRequestedOutput


In [47]:
def predict_triton(texts, model_name="text_classifier_ensemble", url="localhost:8001"):
    """Отправляет запросы к модели через gRPC"""
    
    # Создаем клиент
    client = grpcclient.InferenceServerClient(url=url)
    
    # Проверяем готовность модели
    if not client.is_model_ready(model_name):
        print(f"Модель {model_name} не готова")
        return None
    
    batch_size = len(texts)
    input_tensor = InferInput("TEXT", [batch_size, 1], "BYTES")

    text_data = np.array([[text.encode('utf-8')] for text in texts], dtype=np.object_)
    input_tensor.set_data_from_numpy(text_data)
    
    # Запрашиваем выходные данные
    output_tensor = InferRequestedOutput("OUTPUT")
    
    # Отправляем запрос
    response = client.infer(
        model_name=model_name,
        inputs=[input_tensor],
        outputs=[output_tensor]
    )
    
    # Получаем результаты: форма [batch_size, 1]
    predictions = response.as_numpy("OUTPUT")
    return predictions


In [48]:
def list_models(url="localhost:8001"):
    """Выводит список доступных моделей"""
    try:
        client = grpcclient.InferenceServerClient(url=url)
        models = client.get_model_repository_index()
        print("Доступные модели:")
        for model in models.models:
            status = "✅ ready" if client.is_model_ready(model.name) else "❌ not ready"
            print(f"   • {model.name} - {status}")
    except Exception as e:
        print(f"Ошибка: {e}")

list_models()

Доступные модели:
   • bert_classifier - ✅ ready
   • text_classifier_ensemble - ✅ ready
   • text_tokenizer - ✅ ready


In [49]:
def sigmoid(x):
    """Сигмоида для преобразования logits в вероятность"""
    return 1 / (1 + np.exp(-x))

In [50]:
# Определение тестовых данных
test_texts = [
    "Хорошее место. могу рекомендовать",
    "Хуже места не встечал!",
    "Это место топ 1 с конца по качеству",
    "Капец как плохо в этом заведении",
    "Капец как хорошо в этом заведении",
]

In [51]:
predictions = predict_triton(test_texts)

# Вывод результатов
if predictions is not None:

    for i, (text, pred) in enumerate(zip(test_texts, predictions)):
        # Получаем logit (выход модели до сигмоиды)
        logit = float(pred[0]) if isinstance(pred, np.ndarray) else float(pred)
        
        # Применяем сигмоиду для получения вероятности
        probability = sigmoid(logit)
        
        if probability > 0.5:
            sentiment = "отрицательный отзыв"
            emoji = "❌"
        else:
            sentiment = "положительный отзыв"
            emoji = "✅"
        
        print(f"\n{emoji} Текст {i+1}:")
        print(f"   \"{text}\"")
        print(f"   Logit: {logit:.4f}")
        print(f"   Вероятность (sigmoid): {probability:.4f}")
        print(f"   {sentiment}")
    


✅ Текст 1:
   "Хорошее место. могу рекомендовать"
   Logit: -1.9485
   Вероятность (sigmoid): 0.1247
   положительный отзыв

❌ Текст 2:
   "Хуже места не встечал!"
   Logit: 0.8212
   Вероятность (sigmoid): 0.6945
   отрицательный отзыв

✅ Текст 3:
   "Это место топ 1 с конца по качеству"
   Logit: -1.6474
   Вероятность (sigmoid): 0.1615
   положительный отзыв

❌ Текст 4:
   "Капец как плохо в этом заведении"
   Logit: 1.4386
   Вероятность (sigmoid): 0.8082
   отрицательный отзыв

✅ Текст 5:
   "Капец как хорошо в этом заведении"
   Logit: -0.7744
   Вероятность (sigmoid): 0.3155
   положительный отзыв
